# HW2: Benchmark
## MML1 Projekt: Predikce doby průchodu Mythic+ dungeonem (Pit of Saron) podle složení skupiny

## 1. Navázání na DÚ1: Task framing

**Business problém:** U top EU hráčů Mythic+ se skupiny liší v tom, jak rychle dokáží projít dungeonem. Otázka je, zda lze dobu průchodu předpovědět ze složení skupiny.

**Analytická úloha:** Regrese, predikce doby průchodu dungeonem z pre-run proměnných.

**Výzkumná otázka:** V rámci klíče +16, může složení skupiny (class a spec hráčů) předpovědět dobu průchodu Pit of Saron nad rámec toho, co vysvětluje průměrný item level skupiny?

**Jednotka pozorování:** Jeden dokončený Mythic+ run na úrovni +16 (5 hráčů, jeden dungeon).

**Cílová proměnná:** `clear_time_ms`, skutečná doba průchodu dungeonem v milisekundách (post-run).

**Hlavní metrika:** RMSE v sekundách.

**Omezení datasetu:** Všechny runy pocházejí z top-ranked části EU populace přes rankings endpoint raider.io. Výsledky nelze zobecňovat na běžné hráče.

## 2. Načtení dat

Data jsou připravena a chronologicky rozdělena notebookem `dataprocessing.ipynb`. Zde pouze načítáme výsledné sady z `data/`.

> **Poznámka:** Před spuštěním tohoto notebooku je nutné nejprve spustit `dataprocessing.ipynb`, který vytvoří soubory `data/train.csv`, `data/validation.csv` a `data/test.csv`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

train = pd.read_csv(Path("data/train.csv"))
val   = pd.read_csv(Path("data/validation.csv"))

print(f"Train:      {len(train)} runů")
print(f"Validation: {len(val)} runů")

Train:      6111 runů
Validation: 1309 runů


## 3. Cílová proměnná a metrika

**Cílová proměnná:** `clear_time_ms`, skutečná doba průchodu dungeonem v milisekundách.

**Hlavní metrika:** RMSE (Root Mean Squared Error) přepočítaná na sekundy. Penalizuje velké odchylky více než MAE.

**Vedlejší metrika:** MAE (Mean Absolute Error) v sekundách, průměrná chyba predikce.

Všechny benchmarky jsou vyhodnoceny pouze na validační sadě. Testovací sada nebyla použita.

In [2]:
TARGET    = "clear_time_ms"
ILVL_COLS = ["tank_ilvl", "healer_ilvl", "dps_1_ilvl", "dps_2_ilvl", "dps_3_ilvl"]

results = {}

def evaluate(name: str, y_true, y_pred) -> None:
    rmse = np.sqrt(mean_squared_error(y_true, y_pred)) / 1000
    mae  = mean_absolute_error(y_true, y_pred) / 1000
    results[name] = {"RMSE (s)": round(rmse, 1), "MAE (s)": round(mae, 1)}
    print(f"{name:<45}  RMSE: {rmse:.1f} s   MAE: {mae:.1f} s")

## 4. Naivní baseline: celkový průměr

Pro každý run ve validační sadě je predikcí průměrná hodnota `clear_time_ms` vypočítaná z celé trénovací sady.

**Proč celkový průměr:** Po filtru na +16 je obtížnost konstantní, průměr tedy reprezentuje typický čas průchodu pro tuto úroveň. Jde o nejjednodušší možnou predikci bez jakékoliv informace o složení skupiny nebo ilvl hráčů.

In [3]:
overall_mean = train[TARGET].mean()
y_val        = val[TARGET]
y_pred_naive = pd.Series(overall_mean, index=val.index)

evaluate("Naivní baseline (celkový průměr)", y_val, y_pred_naive)

Naivní baseline (celkový průměr)               RMSE: 90.2 s   MAE: 71.6 s


## 5. ML Benchmark: Ridge regrese (průměrný ilvl)

**Použité features:**
- `avg_group_ilvl` (průměrný item level všech 5 hráčů)

`mythic_level` a `keystone_time_ms` jsou po filtru na +16 konstanty, tedy je vyřazujeme. Benchmark testuje, zda průměrný ilvl skupiny předpoví dobu průchodu lépe než prostý průměr. Složení skupiny (class/spec) opět záměrně není použito, to přijde v navazujícím modelování.

**Preprocessing:** StandardScaler fitovaný pouze na trénovací sadě.

In [4]:
def make_features(df: pd.DataFrame) -> pd.DataFrame:
    X = pd.DataFrame(index=df.index)
    for col in ILVL_COLS:
        X[col] = pd.to_numeric(df[col], errors="coerce")
    X["avg_group_ilvl"] = X[ILVL_COLS].mean(axis=1)
    return X[["avg_group_ilvl"]]

X_train = make_features(train)
X_val   = make_features(val)
y_train = train[TARGET]

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)

ridge = Ridge(alpha=1.0, random_state=42)
ridge.fit(X_train_sc, y_train)
y_pred_ridge = ridge.predict(X_val_sc)

evaluate("Ridge regrese (avg_group_ilvl)", y_val, y_pred_ridge)

Ridge regrese (avg_group_ilvl)                 RMSE: 89.2 s   MAE: 71.3 s


### Koeficienty Ridge modelu

In [5]:
coef_df = pd.DataFrame({
    "Feature":    X_train.columns,
    "Koeficient": ridge.coef_.round(1)
}).sort_values("Koeficient", key=abs, ascending=False)

print(coef_df.to_string(index=False))
print("\nZáporný koeficient: vyšší průměrný ilvl skupiny odpovídá kratšímu času průchodu.")

       Feature  Koeficient
avg_group_ilvl     -6184.3

Záporný koeficient: vyšší průměrný ilvl skupiny odpovídá kratšímu času průchodu.


## 6. Srovnání výsledků

Obě metriky jsou vyhodnoceny výhradně na **validační sadě**. Testovací sada nebyla v tomto úkolu použita.

In [6]:
results_df = pd.DataFrame(results).T
print(results_df.to_string())

                                  RMSE (s)  MAE (s)
Naivní baseline (celkový průměr)      90.2     71.6
Ridge regrese (avg_group_ilvl)        89.2     71.3


## 7. Shrnutí

Ridge regrese s `avg_group_ilvl` překonává naivní baseline o 1 s (RMSE 89.2 vs 90.2). Průměrný item level skupiny tedy v rámci +16 nese malý ale reálný signál, lepší gear = rychlejší průchod. Výchozí laťka pro navazující modelování je **RMSE ≈ 89 s** na validační sadě.

**Metodicky důležité:**
Chronologický split je klíčový, náhodný split by propustil informaci o týdenních afixech mezi sadami. StandardScaler byl fitován pouze na train sadě a aplikován na validation, bez leakage.

**Příští krok:**
Přidat composition features (class, spec na každé roli) a otestovat, zda složení skupiny přináší prediktivní hodnotu nad rámec průměrného ilvl.